In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

from utils import column_encoder

In [346]:
data = pd.read_csv('personalized_recommendation_dataset.csv')
data

,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location
0,User_913,Item_52,Movies,2.0,2023-05-15,369.55,Web,Africa
1,User_3457,Item_66,Electronics,1.4,2023-08-19,255.15,Web,Africa
2,User_1629,Item_1467,Sports,2.7,2024-03-27,296.69,Web,Europe
3,User_3463,Item_697,Movies,1.6,2023-12-03,55.59,Tablet,North America
4,User_2941,Item_1736,Games,3.4,2023-02-06,366.22,Web,South America
...,...,...,...,...,...,...,...,...
149995,User_577,Item_1195,Sports,2.4,2023-10-24,182.04,Web,Europe
149996,User_4996,Item_1335,Books,3.4,2023-07-21,345.84,Mobile App,Europe
149997,User_2804,Item_1956,Games,1.6,2024-02-27,203.77,Mobile App,Europe
149998,User_1443,Item_397,Books,1.4,2024-01-23,475.06,Smart TV,Europe


In [347]:
print(data.info())
print(data.describe())
print(data.isnull().sum())
print(data.nunique())
print(f'Number Of Duplicates: {data.duplicated().sum()}')

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   User_ID    150000 non-null  str    
 1   Item_ID    150000 non-null  str    
 2   Category   150000 non-null  str    
 3   Rating     150000 non-null  float64
 4   Timestamp  150000 non-null  str    
 5   Price      150000 non-null  float64
 6   Platform   150000 non-null  str    
 7   Location   150000 non-null  str    
dtypes: float64(2), str(6)
memory usage: 9.2 MB
None
              Rating          Price
count  150000.000000  150000.000000
mean        2.994918     252.312944
std         1.153429     142.754120
min         1.000000       5.000000
25%         2.000000     128.817500
50%         3.000000     252.610000
75%         4.000000     375.630000
max         5.000000     500.000000
User_ID      0
Item_ID      0
Category     0
Rating       0
Timestamp    0
Price        0
Platform     0
Location 

# Preprocessing Data

In [348]:
def column_encoder(dataset, *cols):
    encoders = {}
    
    for col in cols:
        encoder = LabelEncoder()
        dataset[col] = encoder.fit_transform(dataset[col])
        encoders[col] = encoder
        
    return dataset, encoders

columns = ['User_ID', 'Item_ID', 'Category', 'Platform', 'Location']

data, _ = column_encoder(data, *columns)
data


,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location
0,4905,1468,6,2.0,2023-05-15,369.55,3,0
1,2731,1623,3,1.4,2023-08-19,255.15,3,0
2,700,520,8,2.7,2024-03-27,296.69,3,3
3,2738,1664,6,1.6,2023-12-03,55.59,2,4
4,2158,819,4,3.4,2023-02-06,366.22,3,5
...,...,...,...,...,...,...,...,...
149995,4531,218,8,2.4,2023-10-24,182.04,3,3
149996,4440,374,1,3.4,2023-07-21,345.84,0,3
149997,2006,1063,4,1.6,2024-02-27,203.77,0,3
149998,494,1331,1,1.4,2024-01-23,475.06,1,3



## Transforming Timestamp data
### to Days Passed since interaction

In [349]:
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values('Timestamp')

In [350]:
time_delta = pd.Timestamp.now() - data['Timestamp']
time_delta = np.round(time_delta / pd.Timedelta(days=1), 0)

data['Days_since_interaction'] = time_delta

data

,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location,Days_since_interaction
10773,3418,706,6,4.6,2022-12-24,175.34,3,0,1315.0
97910,2707,1051,8,4.8,2022-12-24,487.57,2,5,1315.0
13563,2962,634,3,1.6,2022-12-24,138.19,0,2,1315.0
34060,4674,363,1,4.1,2022-12-24,290.17,2,3,1315.0
34589,4647,886,5,3.3,2022-12-24,197.78,0,4,1315.0
...,...,...,...,...,...,...,...,...,...
64407,475,448,6,4.0,2024-12-23,155.66,3,3,585.0
68344,3243,1855,6,2.5,2024-12-23,141.37,2,3,585.0
144945,647,283,8,3.9,2024-12-23,435.80,3,2,585.0
29426,2974,30,7,1.8,2024-12-23,129.07,1,3,585.0


# Splitting Data

In [351]:
split_index = int(len(data) * 0.8)

train_data = data.iloc[:split_index].copy()
test_data = data.iloc[split_index:].copy()

In [352]:
price_scaler = StandardScaler()

train_data['Price'] = price_scaler.fit_transform(train_data['Price'].to_numpy().reshape(-1 ,1))
test_data['Price'] = price_scaler.transform(test_data['Price'].to_numpy().reshape(-1 ,1))

In [353]:
print(f'Train shape: {train_data.shape}')
print(f'Test shape: {test_data.shape}')

print(f'\nTrain Data Time Length: \n{train_data['Timestamp'].min()} --> {train_data['Timestamp'].max()}')
print(f'\nTest Data Time Length: \n{test_data['Timestamp'].min()} --> {test_data['Timestamp'].max()}')

Train shape: (120000, 9)
Test shape: (30000, 9)

Train Data Time Length: 
2022-12-24 00:00:00 --> 2024-07-30 00:00:00

Test Data Time Length: 
2024-07-30 00:00:00 --> 2024-12-23 00:00:00


In [354]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from utils import column_encoder

In [355]:
data = pd.read_csv('personalized_recommendation_dataset.csv')
data

,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location
0,User_913,Item_52,Movies,2.0,2023-05-15,369.55,Web,Africa
1,User_3457,Item_66,Electronics,1.4,2023-08-19,255.15,Web,Africa
2,User_1629,Item_1467,Sports,2.7,2024-03-27,296.69,Web,Europe
3,User_3463,Item_697,Movies,1.6,2023-12-03,55.59,Tablet,North America
4,User_2941,Item_1736,Games,3.4,2023-02-06,366.22,Web,South America
...,...,...,...,...,...,...,...,...
149995,User_577,Item_1195,Sports,2.4,2023-10-24,182.04,Web,Europe
149996,User_4996,Item_1335,Books,3.4,2023-07-21,345.84,Mobile App,Europe
149997,User_2804,Item_1956,Games,1.6,2024-02-27,203.77,Mobile App,Europe
149998,User_1443,Item_397,Books,1.4,2024-01-23,475.06,Smart TV,Europe


In [356]:
print(data.info())
print(data.describe())
print(data.isnull().sum())
print(data.nunique())
print(f'Number Of Duplicates: {data.duplicated().sum()}')

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   User_ID    150000 non-null  str    
 1   Item_ID    150000 non-null  str    
 2   Category   150000 non-null  str    
 3   Rating     150000 non-null  float64
 4   Timestamp  150000 non-null  str    
 5   Price      150000 non-null  float64
 6   Platform   150000 non-null  str    
 7   Location   150000 non-null  str    
dtypes: float64(2), str(6)
memory usage: 9.2 MB
None
              Rating          Price
count  150000.000000  150000.000000
mean        2.994918     252.312944
std         1.153429     142.754120
min         1.000000       5.000000
25%         2.000000     128.817500
50%         3.000000     252.610000
75%         4.000000     375.630000
max         5.000000     500.000000
User_ID      0
Item_ID      0
Category     0
Rating       0
Timestamp    0
Price        0
Platform     0
Location 

# Preprocessing Data

In [357]:
def column_encoder(dataset, *cols):
    encoders = {}
    
    for col in cols:
        encoder = LabelEncoder()
        dataset[col] = encoder.fit_transform(dataset[col])
        encoders[col] = encoder
        
    return dataset, encoders

columns = ['User_ID', 'Item_ID', 'Category', 'Platform', 'Location']

data, _ = column_encoder(data, *columns)
data


,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location
0,4905,1468,6,2.0,2023-05-15,369.55,3,0
1,2731,1623,3,1.4,2023-08-19,255.15,3,0
2,700,520,8,2.7,2024-03-27,296.69,3,3
3,2738,1664,6,1.6,2023-12-03,55.59,2,4
4,2158,819,4,3.4,2023-02-06,366.22,3,5
...,...,...,...,...,...,...,...,...
149995,4531,218,8,2.4,2023-10-24,182.04,3,3
149996,4440,374,1,3.4,2023-07-21,345.84,0,3
149997,2006,1063,4,1.6,2024-02-27,203.77,0,3
149998,494,1331,1,1.4,2024-01-23,475.06,1,3



## Transforming Timestamp data
### to Days Passed since interaction

In [358]:
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values('Timestamp')

In [359]:
price_scaler = StandardScaler()

train_data['Price'] = price_scaler.fit_transform(train_data['Price'].to_numpy().reshape(-1 ,1))
test_data['Price'] = price_scaler.transform(test_data['Price'].to_numpy().reshape(-1 ,1))

# Using Basic Collabrative Filtering 

In [360]:
# train_collabrative
train = train_data[['User_ID', 'Item_ID', 'Rating']]
test = test_data[['User_ID', 'Item_ID', 'Rating']]


In [361]:
num_users = data['User_ID'].nunique()
num_items = data['Item_ID'].nunique()

print("Number of users:", num_users)
print("Number of items:", num_items)

Number of users: 5000
Number of items: 2000


In [362]:
# Train Dataset
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
        'User_ID' : train['User_ID'].values,
        'Item_ID' : train['Item_ID'].values
    },
    train['Rating'].values
    )
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
        'User_ID' : test['User_ID'].values,
        'Item_ID' : test['Item_ID'].values
    },
    test['Rating'].values
    )
)



In [363]:
train_ds = train_ds.shuffle(10_000).batch(256)
test_ds = test_ds.batch(256)

In [364]:
class RecommenderModel(tf.keras.Model):
    """
    Simple collaborative filtering model for predicting user-item ratings.

    Each user and item is represented by a learned embedding vector.
    The dot product between those vectors gives the main prediction,
    with separate bias terms for users and items.
    """

    def __init__(self, num_users, num_items, feature_dim=10):
        super().__init__()

        # Learn a feature vector for every user and item
        self.user_features = tf.keras.layers.Embedding(num_users, feature_dim)
        self.item_features = tf.keras.layers.Embedding(num_items, feature_dim)

        # Some users tend to rate higher/lower, and some items are
        # generally rated higher/lower, so we give both their own bias
        self.user_bias = tf.keras.layers.Embedding(num_users, 1)
        self.item_bias = tf.keras.layers.Embedding(num_items, 1)

    def call(self, inputs):
        # Get the user and item IDs from the input dictionary
        user_id = inputs['User_ID']
        item_id = inputs['Item_ID']

        # Look up the learned feature vectors for this user and item
        user_vector = self.user_features(user_id)
        item_vector = self.item_features(item_id)
        # Get the bias values for this user and item
        user_bias = self.user_bias(user_id)
        item_bias = self.item_bias(item_id)

        # Measure how well the user and item vectors match
        dot_product = tf.reduce_sum(
            user_vector * item_vector, axis=1, keepdims=True
        )

        # Combine the vector similarity with the user/item biases
        prediction = dot_product + user_bias + item_bias

        # Remove the extra dimension added by the embedding output
        return tf.squeeze(prediction, axis=1)

In [371]:
model = RecommenderModel(
    num_users=num_users,
    num_items=num_items,
    feature_dim=10,
)

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss = tf.keras.losses.MeanSquaredError()   
)

In [372]:
history = model.fit(train_ds, epochs=10, validation_data=test_ds)

Epoch 1/10


469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 9.5982 - val_loss: 8.8798
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 8.0046 - val_loss: 6.8832
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 5.2000 - val_loss: 3.6582
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 2.5088 - val_loss: 1.8636
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 1.5164 - val_loss: 1.4561
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.3250 - val_loss: 1.4030
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.2878 - val_loss: 1.4025
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.2745 - val_loss: 1.4080
Epoch 9/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.2658 - val_loss: 1.4132
Epoch 10/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.2576 - val_loss: 1.4178


In [373]:
print(history.history)

{'loss': [9.598174095153809, 8.004647254943848, 5.199956893920898, 2.508837938308716, 1.5163780450820923, 1.324971079826355, 1.287787675857544, 1.2744834423065186, 1.2657746076583862, 1.2576498985290527], 'val_loss': [8.879754066467285, 6.883177757263184, 3.6582300662994385, 1.863637924194336, 1.4561158418655396, 1.4030461311340332, 1.402512788772583, 1.4079511165618896, 1.4131661653518677, 1.417772650718689]}
